# Personalized PageRank (PPR) Exploration

Exploration of attack-graph risk scoring using Personalized PageRank on a representative AWS attack chain.

In [ ]:
import sys
from pathlib import Path
import networkx as nx
import pandas as pd

# Ensure repository root is on sys.path
repo_root = Path.cwd().resolve()
if repo_root.name == 'notebooks':
    repo_root = repo_root.parent.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from risk.pagerank import score_digraph

def build_toy_graph() -> nx.DiGraph:
    G = nx.DiGraph()
    G.add_node('S', kind='key', label='S')
    G.add_node('U', kind='user', label='U')
    G.add_node('POL', kind='policy', label='POL')
    G.add_node('STM', kind='stmt', label='STM')
    G.add_node('ADMIN_EQUIV', kind='admin', label='ADMIN_EQUIV')
    G.add_node('BUCKET', kind='bucket', label='BUCKET', personal_data=True)

    G.add_edge('S', 'U', p=0.9, etype='USES_KEY')
    G.add_edge('U', 'POL', p=1.0, etype='HAS_POLICY')
    G.add_edge('POL', 'STM', p=1.0, etype='HAS_STATEMENT')
    G.add_edge('STM', 'ADMIN_EQUIV', p=0.85, etype='CAN_ESCALATE')
    G.add_edge('STM', 'BUCKET', p=0.8, etype='CAN_ACCESS')
    return G

G = build_toy_graph()
print(f'Constructed toy graph with {len(G.nodes())} nodes and {len(G.edges())} edges.')

In [ ]:
# (1) Print scores with entry_nodes=['S']
scores_baseline = score_digraph(G, entry_nodes=['S'])
print('Baseline PPR scores (entry_nodes=["S"], damping=0.85):')
for node, score in scores_baseline.items():
    print(f'  {node:<12}: {score:.6f}')

The baseline PPR scores reflect the reachable attack chain starting from exposed key S, with scores naturally decreasing along each probabilistic traversal step. Nodes further down the path (ADMIN_EQUIV and BUCKET) receive scores proportional to cumulative path transition probabilities.

In [ ]:
# (2) Change edge S->U p from 0.9 to 0.3 and show how the scores change
G_attenuated = build_toy_graph()
G_attenuated['S']['U']['p'] = 0.3

scores_attenuated = score_digraph(G_attenuated, entry_nodes=['S'])

print(f"{'Node':<12} {'Original (p=0.9)':<18} {'Modified (p=0.3)':<18} {'Delta':<12}")
print('-' * 62)
for node in scores_baseline:
    orig = scores_baseline[node]
    mod = scores_attenuated[node]
    delta = mod - orig
    print(f"{node:<12} {orig:<18.6f} {mod:<18.6f} {delta:<+12.6f}")

Lowering the traversal probability on the initial S->U edge directly reduces probability flow to all downstream nodes, because the remaining 70% of transition weight is trapped in the synthetic absorb sink. The source node S itself increases slightly in relative retained score as less flow leaks through the chain.

In [ ]:
# (3) Table of scores at damping 0.5, 0.85, 0.95
damping_values = [0.5, 0.85, 0.95]
table_data = {'Node': list(G.nodes())}

for d in damping_values:
    scores = score_digraph(G, entry_nodes=['S'], damping=d)
    table_data[f'Damping {d}'] = [scores[n] for n in table_data['Node']]

df_damping = pd.DataFrame(table_data)
print(df_damping.to_string(index=False))

Higher damping factors increase the probability of continuing random walks along the attack graph, allowing probability mass to propagate deeper to critical assets like ADMIN_EQUIV and BUCKET. Conversely, lower damping factors force frequent teleports back to entry node S, heavily suppressing downstream risk scores.